# berts-gg-qa-submit

Offline **submit-only** notebook.

Loads 3 SWA model weights from training output and writes `submission.csv`.

In [ ]:
import os, gc, json, html, re
from dataclasses import dataclass
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

In [ ]:
@dataclass
class CFG:
    max_len_q: int = 256
    max_len_a: int = 256
    valid_bs: int = 8
    num_workers: int = 2
    out_dir: str = '/kaggle/working/ggqa_bert_models'
    model_zoo = {
        'roberta_base': '/kaggle/input/datasets/abhishek/roberta-base',
        'roberta_large': '/kaggle/input/datasets/marshal02/robertalarge',
        'deberta_v3_base': '/kaggle/input/datasets/debarshichanda/debertav3base'
    }
cfg=CFG()
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
puncts = [',', '.', '"', ':', ')', '(', '-', '!', '?', '|', ';', "'", '$', '&', '/', '[', ']', '>', '%', '=', '#', '*', '+', '\\', '•', '~', '@', '£', '·', '_', '{', '}', '©', '^', '®', '`', '<', '→', '°', '€', '™', '♥', '←', '×', '§', '…', '\n', '\xa0', '\t', '“', '”', '–', '—']
mispell_dict = {"can't": "cannot", "dont": "do not", "don't": "do not", "doesnt": "does not", "doesn't": "does not", "isn't": "is not", "aren't": "are not", "it's": "it is", "i'm": "i am", "you're": "you are", "they're": "they are"}
def clean_text(x):
    x=str(x)
    for p in puncts: x=x.replace(p, f' {p} ')
    return x
def clean_numbers(x):
    x = re.sub('[0-9]{5,}', '#####', x); x = re.sub('[0-9]{4}', '####', x); x = re.sub('[0-9]{3}', '###', x); x = re.sub('[0-9]{2}', '##', x)
    return x
def _get_mispell(d): return d, re.compile('(%s)' % '|'.join(map(re.escape, d.keys())))
def replace_typical_misspell(text):
    d, rp = _get_mispell(mispell_dict)
    return rp.sub(lambda m: d[m.group(0)], text)
def clean_data(df, cols):
    for c in cols:
        df[c]=df[c].astype(str).map(html.unescape).apply(clean_numbers).str.lower().apply(clean_text).apply(replace_typical_misspell)
    return df

In [ ]:
sample=pd.read_csv('/kaggle/input/competitions/google-quest-challenge/sample_submission.csv')
test=pd.read_csv('/kaggle/input/competitions/google-quest-challenge/test.csv')
text_cols=['question_title','question_body','answer']
test[text_cols]=test[text_cols].fillna('')
test=clean_data(test,text_cols)

with open(f'{cfg.out_dir}/artifacts/target_cols.json','r') as f: target_cols=json.load(f)
with open(f'{cfg.out_dir}/artifacts/label_values.json','r') as f: label_values=json.load(f)
with open(f'{cfg.out_dir}/artifacts/clippings.json','r') as f: CLIPPINGS=json.load(f)
if os.path.exists(f'{cfg.out_dir}/artifacts/model_paths.json'):
    with open(f'{cfg.out_dir}/artifacts/model_paths.json','r') as f: model_paths=json.load(f)
else:
    model_paths={k:f'{cfg.out_dir}/checkpoints/{k}_swa.pt' for k in cfg.model_zoo}
print(model_paths)

In [ ]:
def apply_clipping(preds, cols):
    p=preds.copy()
    for j,c in enumerate(cols):
        if c in CLIPPINGS: p[:,j]=np.clip(p[:,j], float(CLIPPINGS[c][0]), float(CLIPPINGS[c][1]))
    return p
def snap_to_known_labels(preds, cols, lv):
    out=np.zeros_like(preds)
    for j,c in enumerate(cols):
        vals=np.array(lv[c], dtype=np.float32)
        idx=np.abs(preds[:,[j]]-vals.reshape(1,-1)).argmin(axis=1)
        out[:,j]=vals[idx]
    return out
def ensure_non_constant_columns(preds):
    out=preds.copy(); n=len(out)
    tiny=np.linspace(0.0,1e-7,n,dtype=np.float32)
    for j in range(out.shape[1]):
        if np.allclose(out[:,j], out[0,j]): out[:,j]=out[:,j]+tiny
    return out

In [ ]:
class QADataset(Dataset):
    def __init__(self, df, tok, max_len_q=256, max_len_a=256):
        self.df=df; self.tok=tok; self.max_len_q=max_len_q; self.max_len_a=max_len_a
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        qtxt=f"{r['question_title']} [SEP] {r['question_body']}"
        atxt=f"{r['question_title']} [SEP] {r['answer']}"
        q=self.tok(qtxt, truncation=True, max_length=self.max_len_q, padding='max_length', return_tensors='pt')
        a=self.tok(atxt, truncation=True, max_length=self.max_len_a, padding='max_length', return_tensors='pt')
        return {'q_input_ids':q['input_ids'].squeeze(0),'q_attention_mask':q['attention_mask'].squeeze(0),'a_input_ids':a['input_ids'].squeeze(0),'a_attention_mask':a['attention_mask'].squeeze(0)}

class TwinEnsembleRegressor(nn.Module):
    def __init__(self, model_path, n_targets, q_out, a_out):
        super().__init__()
        self.q_encoder=AutoModel.from_pretrained(model_path, local_files_only=True)
        self.a_encoder=AutoModel.from_pretrained(model_path, local_files_only=True)
        h=self.q_encoder.config.hidden_size
        self.head_a=nn.Sequential(nn.Linear(h*2,h), nn.ReLU(), nn.Dropout(0.1), nn.Linear(h,n_targets))
        self.head_q=nn.Sequential(nn.Linear(h,h), nn.ReLU(), nn.Dropout(0.1), nn.Linear(h,q_out))
        self.head_a_only=nn.Sequential(nn.Linear(h,h), nn.ReLU(), nn.Dropout(0.1), nn.Linear(h,a_out))
        self.fusion=nn.Sequential(nn.Linear(h*4,h), nn.ReLU(), nn.Dropout(0.1), nn.Linear(h,n_targets))
    def pooled(self,encoder,ids,mask):
        out=encoder(input_ids=ids, attention_mask=mask).last_hidden_state
        m=mask.unsqueeze(-1)
        return (out*m).sum(1)/m.sum(1).clamp(min=1)
    def forward(self,q_ids,q_mask,a_ids,a_mask):
        qv=self.pooled(self.q_encoder,q_ids,q_mask); av=self.pooled(self.a_encoder,a_ids,a_mask)
        pred_a=self.head_a(torch.cat([qv,av],dim=1))
        pred_b=torch.cat([self.head_q(qv),self.head_a_only(av)],dim=1)
        pred_c=self.fusion(torch.cat([qv,av,torch.abs(qv-av),qv*av],dim=1))
        return 0.25*pred_a + 0.25*pred_b + 0.5*pred_c

In [ ]:
test_predictions=[]
for model_name, model_path in cfg.model_zoo.items():
    print('\n=== infer', model_name, '===')
    tok=AutoTokenizer.from_pretrained(model_path, local_files_only=True)
    ds=QADataset(test.reset_index(drop=True), tok, cfg.max_len_q, cfg.max_len_a)
    dl=DataLoader(ds,batch_size=cfg.valid_bs,shuffle=False,num_workers=cfg.num_workers,pin_memory=True)
    q_out=sum(c.startswith('question_') for c in target_cols)
    a_out=sum(c.startswith('answer_') for c in target_cols)
    model=TwinEnsembleRegressor(model_path, len(target_cols), q_out, a_out).to(device)
    state=torch.load(model_paths[model_name], map_location=device)
    model.load_state_dict(state)
    model.eval()
    pred=[]
    with torch.no_grad():
        for b in dl:
            q_ids=b['q_input_ids'].to(device); q_m=b['q_attention_mask'].to(device)
            a_ids=b['a_input_ids'].to(device); a_m=b['a_attention_mask'].to(device)
            pred.append(model(q_ids,q_m,a_ids,a_m).detach().cpu().numpy())
    pred=np.concatenate(pred)
    test_predictions.append(pred)
    del model, dl, ds; gc.collect(); torch.cuda.empty_cache()

ensemble_test=np.mean(np.stack(test_predictions, axis=0), axis=0)
final_test=snap_to_known_labels(apply_clipping(ensemble_test, target_cols), target_cols, label_values)
final_test=ensure_non_constant_columns(final_test)
submission=sample.copy()
submission[target_cols]=final_test
submission.to_csv('submission.csv', index=False)
print('saved submission.csv')
submission.head()